In [1]:
import numpy as np

In [8]:
import pandas as pd
import re

# --- Config ---
filename = "C:\\Users\\Aishwary Sharma\\Downloads\\Unique_metabolites_416.csv"   # change to your actual CSV file path

# --- Load the data ---
try:
    df = pd.read_csv(filename, encoding='utf-8')
except UnicodeDecodeError:
    print("⚠️ UTF-8 decoding failed, trying latin1...")
    df = pd.read_csv(filename, encoding='latin1')

# --- Regex to detect special/non-ASCII characters ---
special_char_pattern = re.compile(r"[^\x20-\x7E]")

# --- Scan all cells for special characters ---
print("🔍 Scanning for special characters...\n")
found_any = False

for col in df.columns:
    for idx, val in df[col].astype(str).items():
        if special_char_pattern.search(val):
            print(f"Row {idx}, Column '{col}': {val}")
            found_any = True

if not found_any:
    print("✅ No special characters found.")
else:
    print("\n✅ Scan complete — special characters detected above.")


🔍 Scanning for special characters...

✅ No special characters found.


In [7]:
df

,classification,Metabolites,Notation
0,lipid_class,?-Galactosylceramide,?-D-GalCer
1,lipid_class,Cardiolipin,CL
2,lipid_class,Chylomicron-remnants(CE),Chylomicron-remnants(CE)
3,lipid_class,Digalactosylceramide(Gal2Cer),Gal2Cer
4,lipid_class,Free-choelsterol(FC),FC
...,...,...,...
410,nonlipid,Phosphoethanolamine (PETA),PETA
411,nonlipid,Succinate,Succinate
412,nonlipid,UDP-alpha-D-galactose,UDP-Gal
413,nonlipid,UDP-galactose(UDP-Gal),UDP-Gal


In [34]:
import os
import pandas as pd

dir_path = "D:/Raylab/LiMeNEx_Network/src/sbmlData/pathwayTfsModified"

final_df = pd.DataFrame()

for file in os.listdir(dir_path):
    if file.endswith(".csv"):
        file_path = os.path.join(dir_path, file)
        temp_df = pd.read_csv(file_path, dtype=str)
        final_df = pd.concat([final_df, temp_df], ignore_index=True)


In [35]:
# Clean up strings across all columns
for col in final_df.select_dtypes(include='object').columns:
    final_df[col] = final_df[col].astype(str).str.strip().str.replace('\u00A0', ' ', regex=False)


In [37]:
len(final_df)

73505

In [38]:
import pandas as pd
import numpy as np

# --- Step 1: Read and normalize ---
# df = pd.read_csv("your_file.csv", dtype=str)  # read all as string to prevent float conversion

# Clean whitespace and normalize PMIDs
pmid_cols = ["Chea", "Signor", "Trrust"]
for col in pmid_cols:
    final_df[col] = (
        final_df[col]
        .astype(str)
        .str.strip()
        .replace(["nan", "None"], "", regex=False)
        .str.replace(r"\.0$", "", regex=True)  # remove trailing .0
    )

# --- Step 2: Group and merge duplicates ---
def merge_pmids(group):
    """
    For each (TF, TargetGene, Tissue) group:
    - If any row has a PMID in Chea/Signor/Trrust, keep those
    - Else keep first occurrence
    """
    # Keep all rows that have at least one PMID
    with_pmid = group[group[pmid_cols].apply(lambda x: any(x.notna() & (x != "")), axis=1)]
    if not with_pmid.empty:
        # If multiple with PMIDs exist, keep the one with most PMIDs filled
        with_pmid["pmid_count"] = with_pmid[pmid_cols].apply(lambda x: (x != "").sum(), axis=1)
        best_row = with_pmid.sort_values("pmid_count", ascending=False).iloc[0]
        return best_row.drop("pmid_count")
    else:
        return group.iloc[0]

cleaned_df = final_df.groupby(["TF", "TargetGene", "Tissue"], dropna=False, as_index=False).apply(merge_pmids).reset_index(drop=True)

print(f"✅ Cleaned dataset has {len(cleaned_df)} unique rows.")


✅ Cleaned dataset has 66082 unique rows.


C:\Temp\ipykernel_24904\592646567.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cleaned_df = final_df.groupby(["TF", "TargetGene", "Tissue"], dropna=False, as_index=False).apply(merge_pmids).reset_index(drop=True)


In [40]:
cleaned_df.to_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/cleaned_tf_targetgene_tissue_groups.csv", index=False)

In [10]:
import pandas as pd
df = pd.read_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/cleaned_tf_targetgene_tissue_groups.csv", dtype=str)

In [11]:
df.head()

,TF,TargetGene,Tissue,Experiment,Chea,Signor,Trrust
0,125DVD3,ABCA1,colon,CEBPB IP | 125DVD3,NaN,NaN,NaN
1,125DVD3,ABCG1,colon,"CEBPB IP | 125DVD3, CDX2 IP | 125DVD3",NaN,NaN,NaN
2,125DVD3,ABHD3,colon,"CDX2 IP | 125DVD3, CEBPB IP | 125DVD3",NaN,NaN,NaN
3,125DVD3,ACAA1,colon,"CEBPB IP | 125DVD3, CDX2 IP | 125DVD3",NaN,NaN,NaN
4,125DVD3,ACACA,colon,CEBPB IP | 125DVD3,NaN,NaN,NaN


In [ ]:
targetGene = df['TargetGene'].unique().tolist()
len(targetGene)


AttributeError: 'list' object has no attribute 'tolist'

In [7]:
import pickle
with open("targetGene.pkl", "wb") as f:
    pickle.dump(targetGene, f)


In [1]:
import pandas as pd
df = pd.read_csv("D:/Raylab/LiMeNEx_Network/src/sbmlData/cleaned_tf_targetgene_tissue_groups.csv", dtype=str)

In [2]:
mapping = {}


for index, row in df.iterrows():
    tf = row['TF']
    target_gene = row['TargetGene']
    
    if target_gene not in mapping:
        mapping[target_gene] = [tf]
    else:
        if tf not in mapping[target_gene]:
            mapping[target_gene].append(tf)

In [4]:
import json

with open("D:/Raylab/LiMeNEx_Network/src/sbmlData/tf_targetgene_mapping.json", "w") as f:
    json.dump(mapping, f)